# Level 1 — Scientific Problem Framing and Python Foundation

## Problem Statement: HydroSense-Kenya Smart Irrigation System
In the Arid and Semi-Arid Lands (ASALs) of Kenya, agricultural productivity is fundamentally constrained by water availability. Smallholder farmers and demonstration hubs face severe trade-offs between water conservation, crop yield, and the energy costs associated with pumping groundwater or surface reserves. Traditional irrigation scheduling often relies on calendar-based approaches or subjective visual inspection of soil, leading to either critical crop water stress (under-irrigation) or wasted water and excessive drainage (over-irrigation).

The HydroSense-Kenya project aims to address this challenge by developing a computational, climate-aware decision-support system. Using a daily water balance model, the system tracks the physical movement of water into and out of the root zone. The scientific question we are answering is: *Given specific local weather parameters and discrete soil-sensor readings, how can we mathematically model water availability, accurately simulate future soil moisture depletion, and recommend an irrigation strategy that minimizes total water withdrawal without dropping below crop-specific wilting points?*

To achieve this, we rely on the fundamental mass-balance equation of soil moisture:
$S_{t+1} = S_t + R_t - ET_t - D_t + I_t$

Where the primary water loss driver, Evapotranspiration ($ET$), is estimated using our required empirical model based on temperature, wind speed, and solar index. By translating this real-world physics problem into a vectorized Python simulation, we can optimize the timing and volume of irrigation ($I_t$). 

**Assumptions and Limitations of the Initial Model:**
1. We assume the soil compartment is a single homogeneous layer.
2. The empirical ET formula is a localized approximation and does not account for specific aerodynamic crop canopy resistance (like Penman-Monteith does).
3. Runoff is assumed to be negligible for this specific farm's topography.

## Data Dictionary

### 1. weather_daily.csv
* `date`: The calendar date of the reading.
* `rainfall_mm`: Total daily precipitation ($R_t$) in millimeters.
* `temperature_c`: Mean daily temperature in Celsius ($T$).
* `humidity_pct`: Relative humidity percentage.
* `wind_speed_mps`: Wind speed in meters per second ($W$).
* `solar_index`: Normalized solar radiation intensity ($Solar$).

### 2. soil_sensor_data.csv
* `timestamp`: Date and time of the noon reading.
* `zone_id`: The specific crop zone (Zone A, B, or C).
* `soil_moisture_pct`: Volumetric water content of the soil ($S_t$).
* `tank_level_liters`: Water remaining in the primary storage tank.
* `pump_flow_lpm`: The flow rate of the irrigation pump in Liters Per Minute.
* `pump_power_watts`: The energy draw of the pump.
* `sensor_status`: Health of the hardware ('OK' or fault codes).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 1. Load Initial Datasets
repo_root = Path('.').resolve().parent
weather = pd.read_csv(repo_root / 'data' / 'raw' / 'weather_daily.csv', na_values=["NA", ""])
weather['date'] = pd.to_datetime(weather['date'])

print(f"Loaded {len(weather)} days of weather data.")

In [ ]:
# 2. Reusable Python Functions for Core Domain

def calc_empirical_et(temp, wind, solar):
    """
    Computes simplified daily Evapotranspiration (ET) in mm.
    Formula defined in project brief: ET = max(0, 0.12*T + 0.35*W + 2.4*Solar)
    """
    et = 0.12 * temp + 0.35 * wind + 2.4 * solar
    return max(0.0, et)

def calc_water_balance(s_current, rain, et, drainage, irrigation):
    """
    Computes next day's soil moisture based on the mass balance equation.
    """
    s_next = s_current + rain - et - drainage + irrigation
    return max(0.0, s_next)

print("Core mathematical functions defined successfully.")

In [ ]:
# 3. Basic Scientific Plot (Baseline Visualization)
plt.figure(figsize=(10, 4))
plt.bar(weather['date'], weather['rainfall_mm'], color='royalblue')
plt.title("Daily Rainfall (mm) - March 2026")
plt.xlabel("Date")
plt.ylabel("Rainfall (mm)")
plt.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()